### Vector DB and Rag

In [7]:
!pip install chromadb

In [8]:
import chromadb
client = chromadb.Client()

In [9]:
collection = client.get_or_create_collection("news")

In [10]:
collection.add(
    ids = ["id1","id2","id3","id4"],
    documents = [
        "Apple is leading in a smart phone game with iphone sales up by 35%",
        "Tesla booked a minor profit of 1 bilion $ in Q2",
        "Apples are high in fiber , vitamin C , and various antioxidants",
        "SpaceX got NASA contract worth 10 bilion $"
    ]
)

C:\Users\royal\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:18<00:00, 1.05MiB/s]


In [11]:
results = collection.query(
    query_texts=[" i love ipad"],
    n_results=4
)

print(results)

{'ids': [['id1', 'id3', 'id2', 'id4']], 'embeddings': None, 'documents': [['Apple is leading in a smart phone game with iphone sales up by 35%', 'Apples are high in fiber , vitamin C , and various antioxidants', 'Tesla booked a minor profit of 1 bilion $ in Q2', 'SpaceX got NASA contract worth 10 bilion $']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None, None, None]], 'distances': [[1.4372706413269043, 1.6695863008499146, 1.9741275310516357, 2.0362281799316406]]}


In [12]:
results = collection.query(
    query_texts=["This is a query related to Elon Musk"],
    n_results=5
)

print(results)

{'ids': [['id2', 'id4', 'id1', 'id3']], 'embeddings': None, 'documents': [['Tesla booked a minor profit of 1 bilion $ in Q2', 'SpaceX got NASA contract worth 10 bilion $', 'Apple is leading in a smart phone game with iphone sales up by 35%', 'Apples are high in fiber , vitamin C , and various antioxidants']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None, None, None]], 'distances': [[1.453629732131958, 1.53712797164917, 1.745964765548706, 1.787568211555481]]}


In [13]:
results = collection.query(
    query_texts=["this "],
    n_results=4
)

print(results)

{'ids': [['id1', 'id2', 'id4', 'id3']], 'embeddings': None, 'documents': [['Apple is leading in a smart phone game with iphone sales up by 35%', 'Tesla booked a minor profit of 1 bilion $ in Q2', 'SpaceX got NASA contract worth 10 bilion $', 'Apples are high in fiber , vitamin C , and various antioxidants']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None, None, None]], 'distances': [[1.7748842239379883, 1.7785358428955078, 1.886833667755127, 2.1232545375823975]]}


### Rag

In [14]:
!pip install -U tokenizers transformers
!pip install langchain langchain-community langchain-chroma langchain-groq langchain-huggingface pypdf sentence-transformers

In [16]:
# !pip install pypdf
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "/telecom_guide.pdf"

loader = PyPDFLoader(file_path="/telecom_guide.pdf")
pages = loader.load()

print(f"loaded {len(pages)} pages from the PDF.")
print(pages[0].page_content[:500])

ValueError: File path /telecom_guide.pdf is not a valid file or url

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(pages)

print(f"Total chunks: {len(chunks)} (from {len(pages)} pages)")
print(f"Avg chunk length : {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print("\n---Example chunk---\n")
print(chunks[1].page_content)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("loading embedding model ( downloads 90 MB on first run)...")
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Embedding all chunks and storing in chroma (in memory)...")
vector_store = Chroma.from_documents(documents=chunks, embedding=embedding)

print(f"vector store ready . {vector_store._collection.count()} vectors stored.")

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 10})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

print(f"Query : {test_query}\n")
print(f"Retrived {len(retrieved)} chunks : \n")
for i ,doc in enumerate(retrieved,1):
    print(f"---chunk {i} --- ")
    print(doc.page_content[:300])
    print( )